# Build 03-07 · Retuning — recall pinned to production, how much precision comes back

**Kernel: the analysis `.venv`** (`python3`) — reads parquet + CSV only, loads no model.

03_05 tunes every axis to hold precision ≥ `config.TARGET_PRECISION` (0.985). That target is
confirmed (supervisor, 2026-09-28) to be **v3's own requirement, never v2's** — v2 was live at
whatever precision its real deployed tau actually gave, not at 0.985. This notebook asks the
more useful question instead: given that production already runs at some recall, how much
precision can a corrected model buy while not giving that recall up.

**Whole-split only (2026-09-28).** Every anchor and every tuning decision here uses the whole
split, forced labels included, never the garage-verified slice. Garage-verified rows thin out to
almost nothing near or above tau ("positivity is dead at tau"), exactly the region a recall
floor selects on, so a garage-only reading has almost nothing to say about it. The
garage-verified reading of the resulting cutoffs is still computed and saved (`precision_garage`/
`recall_garage` columns) purely as a secondary diagnostic; it drives no selection here.

Mechanically: `threshold.tune(..., recall_min=anchor)` on the SAME clean holdout slice 03_05
uses for `tuned_holdout` (a split neither the fit nor the final eval slice touches), then
measured on the eval slice.

The recall floor itself is **never hardcoded** (see memory: no raw Allianz numbers in code) —
§2b below reads it off baseline's own real whole-split recall at its own real deployed tau(s),
live, every run.

### 실행 전 설정

**커널**: analysis `.venv` (`python3`)

**바꿀 것 (§1)**: `VERSION` — `"v2"` 또는 `"v3"` (`HOLDOUT_SPLIT`은 여기서 자동으로 정해짐: v2는 `val`, v3는 `test`)

**선행 조건**: `03_04_prediction.ipynb`까지 다 돌리고 각 axis의 mitigated score가 `reeval/<v>/`에 있어야 함. `03_05_reevaluation.ipynb`은 선행 필요 없음 (이 노트북은 매번 원시 점수에서 다시 집계함, 03_05의 출력을 읽지 않음).

**whole-split이 유일 기준**: 이제부터 anchor와 tuning 둘 다 whole-split baseline recall만 쓴다 (garage-verified 버전은 사용 안 함, 2026-09-28 결정). garage 측정값은 참고용으로만 남긴다.

**fixed-tau 변이는 모두 제외된다**: `rarity_fixed`/`pnu_fixed`는 논문 본문에서 빼기로 했으니, 이 노트북의 `MODELS` 리스트에서도 같이 뺀다.

In [ ]:
# §0 — setup (analysis .venv kernel)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_auc_score

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle
import threshold

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — RUN_SPEC
VERSION = "v3"        # "v2" works too
TRAIN_SPLIT = "train"                 # unused here (03_05's "tuned"/train-tuned axis is not
                                       # repeated in this notebook — only the honest holdout
                                       # discipline is), kept for symmetry with 03_05's RUN_SPEC
EVAL_SPLIT = config.OOT_SPLIT[VERSION]            # v1/v2 invert "test" — never picked by name
HOLDOUT_SPLIT = {"v2": "val", "v3": "test"}[VERSION]   # a split neither the fit nor EVAL_SPLIT
                                                          # touches — same slice 03_05 uses for
                                                          # tuned_holdout
ID_COL = "claim_id"
SCORE_COL = "model_" + VERSION + "_score"

_rule = config.DECISION_RULES[VERSION]
if _rule["shape"] == "global":
    TAU_GRID = [float(_rule["threshold"])]
elif _rule["shape"] == "piecewise_global":
    TAU_GRID = sorted({float(r["threshold"]) for r in _rule["regimes"]})
else:
    raise SystemExit("segmented rule (v1) is out of scope here")

# v2's regime[0] (0.8915) is not a confirmed documented tau — config.py's own DECISION_RULES['v2']
# comment says "nothing may read regimes[0] as the documented tau". The other two grid values
# (0.872, 0.825) are supervisor-confirmed real production cutoffs — keep only those, so every
# anchor built below sits on a tau that genuinely ran in production.
if VERSION == "v2":
    TAU_GRID = [t for t in TAU_GRID if abs(t - 0.8915) > 1e-9]

REEVAL_DIR = ROOT / "src" / "data" / "real" / "reeval" / VERSION
figstyle.FIG_DIR = ROOT / "figures" / "mitigation" / "03_07" / VERSION
figstyle.ALIAS_SPLIT = False    # see 03_05 §1's note — this notebook never touches feature names
print("eval split:", EVAL_SPLIT, "| holdout split:", HOLDOUT_SPLIT, "| production tau grid:", TAU_GRID)
print("FIG_DIR =", figstyle.FIG_DIR)

In [ ]:
# §2 — assemble: corrector_targets (treatment pre-joined by 03_01) + one score column per model.
# Same shape as 03_05 §2 — kept self-contained rather than imported, matching this repo's
# per-notebook convention (each Build 03 notebook stands alone).
def load_split_frame(split: str) -> pd.DataFrame:
    """corrector_targets of `split` — targets already joined to the recorded treatment (03_01)."""
    p = config.split_path("corrector_targets", VERSION, split)
    assert p.is_file(), f"{p} missing — run 03_01_corrector_inputs first"
    m = pd.read_parquet(p)
    m["date"] = pd.to_datetime(m["date"])
    print(f"{VERSION} {split}: {len(m):,} treated rows (unmatched already dropped in 03_01)")
    return m


def attach_scores(frame: pd.DataFrame, split: str) -> tuple[pd.DataFrame, list[str]]:
    """Merge every available model's scores on `split`: mitigated axes + baseline if exported."""
    models = []
    bpath = config.path("scores", VERSION, split=split)
    if bpath.is_file():
        b = pd.read_parquet(bpath)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: "baseline"})
        frame = frame.merge(b, on=ID_COL, how="left")
        models.append("baseline")
    else:
        print(f"  note: no baseline scores at {bpath} (run src/scoring/predict.py for {VERSION}) — skipped")
    prefix = f"{VERSION}_mitigated_scores_{split}_"
    for p in sorted(REEVAL_DIR.glob(prefix + "*.parquet")):
        tag = p.stem[len(prefix):]
        s = pd.read_parquet(p)[[ID_COL, SCORE_COL]].rename(columns={SCORE_COL: tag})
        frame = frame.merge(s, on=ID_COL, how="left")
        models.append(tag)
    return frame, models


def drop_fixed(models: list[str]) -> list[str]:
    """Paper scope excludes the fixed-tau rarity/pnu variants — see the setup note above."""
    return [m for m in models if not m.endswith("_fixed")]


ev, MODELS = attach_scores(load_split_frame(EVAL_SPLIT), EVAL_SPLIT)
hd, HD_MODELS = attach_scores(load_split_frame(HOLDOUT_SPLIT), HOLDOUT_SPLIT)
MODELS, HD_MODELS = drop_fixed(MODELS), drop_fixed(HD_MODELS)
assert MODELS, "no score files for the eval split — run 03_04 first"

ev_matched = ev["decision"].notna()
ev_gar = ev.loc[ev_matched & (ev["decision"] == 0)]     # kept only for the secondary garage
                                                          # diagnostic columns in §3/§5
print(f"eval garage rows: {len(ev_gar):,}")
print("models:", MODELS)


def load_full_split(split: str) -> tuple[pd.DataFrame, list[str]]:
    """targets(split) — every row with its recorded label — plus one score column per model."""
    t = pd.read_parquet(config.split_path("targets", VERSION, split))[[ID_COL, "observed"]]
    frame, models = attach_scores(t, split)
    return frame, drop_fixed(models)


ev_all, all_models = load_full_split(EVAL_SPLIT)
assert all_models == MODELS, "score files differ between the treated and the full read — rerun §2"
hd_all, hd_all_models = load_full_split(HOLDOUT_SPLIT)
assert hd_all_models == HD_MODELS, (
    "score files differ between the treated and the full read on the holdout split — rerun §2")


def metrics_at(y: np.ndarray, s: np.ndarray, tau: float) -> dict:
    """AUC + precision/recall/CM at the STRICT rule score > tau (score == tau garages)."""
    pred = s > tau
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    tn = int((~pred & (y == 0)).sum()); fn = int((~pred & (y == 1)).sum())
    return {"AUC": float(roc_auc_score(y, s)),
            "precision": tp / (tp + fp) if tp + fp else float("nan"),
            "recall": tp / (tp + fn) if tp + fn else float("nan"),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

### §2b. recall-min 값을 구하는 방식

baseline이 실제로 배포된 tau(`TAU_GRID`, §1)에서 whole split(강제 라벨 포함) 위에서 낸 recall을 그대로 읽어서 floor로 쓴다. garage 기준은 더 쓰지 않는다 (positivity가 tau 근처에서 거의 죽어있어서, garage row가 거의 없기 때문). 값은 매번 코드가 직접 계산해서 출력만 하고, 숫자를 코드에 박아넣지는 않는다.

In [ ]:
# §2b — the recall floor itself: baseline's OWN real recall, at its OWN real deployed tau, on the
# WHOLE split (forced labels in) — the only basis this notebook uses from 2026-09-28 on. Read
# live every run, never hardcoded. One row per grid tau (v2: two eras' cutoffs; v3: one).
if "baseline" not in MODELS:
    raise SystemExit(f"§2b needs baseline scores for {VERSION} — run src/scoring/predict.py first")

a0 = ev_all[[ID_COL, "observed", "baseline"]].dropna(subset=["baseline"])
y0_all = a0["observed"].astype(int).to_numpy()
s0_all = a0["baseline"].to_numpy(dtype=float)

anchor_rows = []
for tau in TAU_GRID:
    ma = metrics_at(y0_all, s0_all, tau)
    anchor_rows.append({"anchor_tau": tau, "baseline_precision_whole": round(ma["precision"], 4),
                         "baseline_recall_whole": round(ma["recall"], 4), "n_whole": len(a0)})
anchors = pd.DataFrame(anchor_rows)
display(anchors)
figstyle.save_table(anchors, f"{VERSION}_0307_sec2b_recall_min_anchors_{EVAL_SPLIT}", index=False)

### §3. 같은 recall을 지키면 precision이 얼마나 돌아오는지

`§2b`의 anchor마다, `§1`의 `HOLDOUT_SPLIT`의 whole-split 읽음(fit도 eval도 안 본 진짜 깨끗한 슬라이스)에서 `threshold.tune(..., recall_min=anchor)`로 tau를 고른다. 선택은 whole-split holdout에서만 하고, 측정은 eval garage(보조 참고)/whole 둘 다에서 한다.

In [ ]:
# §3 — for every (model, anchor), the highest whole-split-holdout tau whose whole-split recall
# still clears the anchor (threshold.tune(..., recall_min=...), STRICT >, see threshold.py's own
# note that this mode's boundary is inherited "as received" from Allianz's own
# select_best_threshold) — same discipline as 03_05's tuned_holdout: chosen on a split the
# model's fit and the final eval slice both never touch, then MEASURED on the eval slice, never
# on the slice it was chosen on. Garage-verified precision/recall of the resulting cutoff is also
# recorded, purely as a secondary diagnostic — it drives no selection here.
rows = []
for name in MODELS:
    if name not in HD_MODELS:
        print(f"  {name}: no scores on {HOLDOUT_SPLIT} — skipped")
        continue
    ha = hd_all[["observed", name]].dropna(subset=[name])
    y_hd = ha["observed"].astype(int).to_numpy()
    s_hd = ha[name].to_numpy(dtype=float)

    g = ev_gar[[ID_COL, "observed", name]].dropna(subset=[name])
    y_gar = g["observed"].astype(int).to_numpy()
    s_gar = g[name].to_numpy(dtype=float)

    a = ev_all[[ID_COL, "observed", name]].dropna(subset=[name])
    y_all = a["observed"].astype(int).to_numpy()
    s_all = a[name].to_numpy(dtype=float)
    s_all_eval = ev[[ID_COL, name]].dropna(subset=[name])   # every scored row — for the decisions file

    for _, arow in anchors.iterrows():
        anchor_tau = float(arow["anchor_tau"])
        anchor_recall = float(arow["baseline_recall_whole"])
        t = threshold.tune(y_hd, s_hd, recall_min=anchor_recall)
        if t is None:
            print(f"  {name}: whole-split holdout recall never exceeds the anchor "
                  f"({anchor_recall:.4f}, from production tau={anchor_tau}) — no cutoff picked")
            rows.append({"model": name, "anchor_tau": anchor_tau, "anchor_recall": anchor_recall,
                         "tau": None, "reached": False})
            continue

        gar_m = metrics_at(y_gar, s_gar, float(t))
        all_m = metrics_at(y_all, s_all, float(t))
        rows.append({"model": name, "anchor_tau": anchor_tau, "anchor_recall": anchor_recall,
                     "tau": round(float(t), 6), "reached": True,
                     "precision_garage": round(gar_m["precision"], 4),
                     "recall_garage": round(gar_m["recall"], 4),
                     "precision_whole": round(all_m["precision"], 4),
                     "recall_whole": round(all_m["recall"], 4),
                     "TP_whole": all_m["TP"], "FP_whole": all_m["FP"],
                     "FN_whole": all_m["FN"], "TN_whole": all_m["TN"]})

        dp = REEVAL_DIR / f"{VERSION}_decisions_{EVAL_SPLIT}_{name}_recallmin{round(anchor_recall, 4)}_tau{round(float(t), 6)}.parquet"
        pd.DataFrame({ID_COL: s_all_eval[ID_COL].to_numpy(),
                      "score": s_all_eval[name].to_numpy(dtype=float),
                      "decision": (s_all_eval[name].to_numpy(dtype=float) > float(t)).astype(int)}
                     ).to_parquet(dp, index=False)

retune = pd.DataFrame(rows)
display(retune)
figstyle.save_table(retune, f"{VERSION}_0307_sec3_retune_metrics_{EVAL_SPLIT}", index=False)

### §4. PR curve(whole split) 위에 생산 recall 위치 표시

점선은 03_05의 0.985 target (subsec B), 수직선은 §2b의 whole-split anchor recall(들)이다.

In [ ]:
# PR curves overlap heavily near the top-right, so: distinct line styles/widths (baseline drawn
# behind, thin dotted), and every PR figure is saved TWICE as separate single-panel images --
# no titles (they live in the LaTeX caption); the full view and a "_zoom" view on precision >= PR_ZOOM_LO where the 0.985 line is readable.
PR_ZOOM_LO = 0.95
PR_STYLES = ["-", "-.", (0, (5, 1.5)), (0, (1, 1.2)), (0, (3, 1, 1, 1))]
PR_WIDTHS = [2.4, 1.9, 1.6, 1.4, 1.2]


PR_CLOSE = 0.04     # recalls closer than this count as crowded (their labels get staggered)


def _mark_recalls(ax, items, zoom):
    """items = [(recall, colour, text)]: a dashed vertical line per item, its value written beside
    it. Labels whose recalls sit close together are staggered in height and alternate sides, on a
    white halo, so they stay readable when the lines crowd together."""
    y0 = PR_ZOOM_LO + 0.002 if zoom else 0.03
    step = 0.008 if zoom else 0.10
    prev_x, level = None, 0
    for r, colour, text in sorted(items, key=lambda t: t[0]):
        ax.axvline(r, ls="--", lw=1.0, color=colour, alpha=0.8, zorder=3)
        level = level + 1 if prev_x is not None and r - prev_x < PR_CLOSE else 0
        ax.text(r, y0 + step * level, text, fontsize=7, color=colour, rotation=90, va="bottom",
                ha="right" if level % 2 == 0 else "left",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.6, alpha=0.75), zorder=11)
        prev_x = r


def draw_pr(frame, models, target, fname, marks=(), mark_recall_at_target=True):
    """marks = [(recall, label, colour, linestyle), ...] vertical recall lines.
    mark_recall_at_target (default True) adds, per model, a dashed vertical line in the curve's own
    colour at the recall where the curve last holds precision >= target, labelled with that recall."""
    for zoom in (False, True):
        fig, ax = plt.subplots(figsize=figstyle.FIG_1)
        slot = 0
        at_target = []
        for name in models:
            d = frame[["observed", name]].dropna(subset=[name])
            y = d["observed"].astype(int).to_numpy()
            prec, rec, _ = precision_recall_curve(y, d[name].to_numpy(dtype=float))
            if name == "baseline":
                (ln,) = ax.plot(rec[:-1], prec[:-1], ls=":", lw=1.2, color=figstyle.NEUTRAL,
                                label="baseline", zorder=1)
            else:
                (ln,) = ax.plot(rec[:-1], prec[:-1], ls=PR_STYLES[slot % len(PR_STYLES)],
                                lw=PR_WIDTHS[slot % len(PR_WIDTHS)], alpha=0.9, label=name,
                                zorder=2)
                slot += 1
            ok = prec[:-1] >= target
            if mark_recall_at_target and ok.any():
                r_t = float(rec[:-1][ok].max())
                at_target.append((r_t, ln.get_color(), f"{r_t:.3f}"))
        ax.axhline(target, ls="--", lw=1.3, color=figstyle.INK, zorder=10)
        _mark_recalls(ax, at_target, zoom)
        for r, label, colour, ls in marks:
            ax.axvline(r, ls=ls, lw=1.2, color=colour, zorder=3)
            ax.text(r, 0.03 if not zoom else PR_ZOOM_LO + 0.002, label, fontsize=7,
                    color=colour, rotation=90, va="bottom", ha="right")
        if zoom:
            ax.set_ylim(PR_ZOOM_LO, 1.005)
        else:
            ax.set_ylim(0, 1.02)
            ax.text(0.01, target + 0.01, f"{target} target", fontsize=8, color=figstyle.INK,
                    zorder=10)
        ax.set_xlabel("recall"); ax.set_ylabel("precision")
        ax.legend(fontsize=8)
        figstyle.save_fig(fig, fname + ("_zoom" if zoom else ""))
        plt.show()


draw_pr(ev_all, MODELS, config.TARGET_PRECISION,
        f"{VERSION}_0307_sec4_retune_pr_{EVAL_SPLIT}_all",
        marks=[(float(a["baseline_recall_whole"]), f"{float(a['baseline_recall_whole']):.3f}",
                figstyle.NEUTRAL, ":") for _, a in anchors.iterrows()])

### §5. v3, v2가 실제로 낸 recall을 기준으로 (v3 only)

v3는 recall이 무너져서 production에 올라가지도 못한 모델이다. v3 자기 recall을 floor로 쓰는 건 이미 무너진 기준을 다시 확인하는 것뿐이라, 대신 v2가 실제로 production에서 낸 whole-split recall을 floor로 쓴다. v2는 이 노트북의 `VERSION`과 무관하게 항상 v2 자신의 실제 배포 tau·score로 직접 읽는다.

In [ ]:
# §5 -- cross-version check (v3 only): what if v3 had to match the recall v2 ACTUALLY ran at in
# production, not v3's own (already collapsed) recall? v2 never had to clear a 0.985 floor and
# stayed live; v3's own attempt collapsed recall before it ever reached production. v2 is read
# here by its OWN version string, tau grid and score column -- never through this notebook's
# VERSION/TAU_GRID/MODELS, which stay v3's own throughout. Whole-split only, same as §2b/§3.
if VERSION != "v3":
    print("§5 only runs for VERSION='v3' -- v2 anchored on itself is circular, skipped here.")
else:
    V2 = "v2"
    v2_eval_split = config.OOT_SPLIT[V2]                       # "test" (v1/v2 invert the name)
    v2_score_col = "model_" + V2 + "_score"
    v2_rule = config.DECISION_RULES[V2]
    v2_tau_grid = sorted({float(r["threshold"]) for r in v2_rule["regimes"]})
    v2_tau_grid = [t for t in v2_tau_grid if abs(t - 0.8915) > 1e-9]   # same exclusion as §1

    def _v2_load(kind: str, split: str) -> pd.DataFrame:
        """v2's own `kind` (targets | corrector_targets) on `split`, plus v2's own baseline
        score. Version-fixed to v2 regardless of this notebook's own VERSION -- v2 is the
        cross-version anchor source here, never the run's own subject."""
        base = pd.read_parquet(config.split_path(kind, V2, split))
        bpath = config.path("scores", V2, split=split)
        assert bpath.is_file(), f"{bpath} missing -- v2 baseline scores are needed for §5"
        b = pd.read_parquet(bpath)[[ID_COL, v2_score_col]].rename(columns={v2_score_col: "baseline"})
        return base.merge(b, on=ID_COL, how="left").dropna(subset=["baseline"])

    v2_all = _v2_load("targets", v2_eval_split)

    v2_anchor_rows = []
    for tau in v2_tau_grid:
        ma = metrics_at(v2_all["observed"].astype(int).to_numpy(),
                         v2_all["baseline"].to_numpy(dtype=float), tau)
        v2_anchor_rows.append({"v2_tau": tau, "anchor_recall": round(ma["recall"], 4)})
    v2_anchors = pd.DataFrame(v2_anchor_rows)
    display(v2_anchors)
    figstyle.save_table(v2_anchors, f"{VERSION}_0307_sec5_v2_anchor_recalls_{EVAL_SPLIT}", index=False)

    v5_rows = []
    for name in MODELS:
        if name not in HD_MODELS:
            print(f"  {name}: no scores on {HOLDOUT_SPLIT} -- skipped")
            continue
        ha = hd_all[["observed", name]].dropna(subset=[name])
        y_hd = ha["observed"].astype(int).to_numpy(); s_hd = ha[name].to_numpy(dtype=float)

        g = ev_gar[[ID_COL, "observed", name]].dropna(subset=[name])
        y_gar = g["observed"].astype(int).to_numpy(); s_gar = g[name].to_numpy(dtype=float)
        a = ev_all[[ID_COL, "observed", name]].dropna(subset=[name])
        y_all = a["observed"].astype(int).to_numpy(); s_all = a[name].to_numpy(dtype=float)

        for _, arow in v2_anchors.iterrows():
            v2_tau = float(arow["v2_tau"]); anchor_recall = float(arow["anchor_recall"])
            t = threshold.tune(y_hd, s_hd, recall_min=anchor_recall)
            if t is None:
                print(f"  {name} (v2 tau={v2_tau}): v3 whole-split holdout recall never exceeds "
                      f"{anchor_recall:.4f} -- no cutoff picked")
                v5_rows.append({"model": name, "v2_tau": v2_tau, "anchor_recall": anchor_recall,
                                 "tau": None, "reached": False})
                continue
            gar_m = metrics_at(y_gar, s_gar, float(t))
            all_m = metrics_at(y_all, s_all, float(t))
            v5_rows.append({"model": name, "v2_tau": v2_tau, "anchor_recall": anchor_recall,
                             "tau": round(float(t), 6), "reached": True,
                             "precision_garage": round(gar_m["precision"], 4),
                             "recall_garage": round(gar_m["recall"], 4),
                             "precision_whole": round(all_m["precision"], 4),
                             "recall_whole": round(all_m["recall"], 4)})

    v2_anchor_retune = pd.DataFrame(v5_rows)
    display(v2_anchor_retune)
    figstyle.save_table(v2_anchor_retune, f"{VERSION}_0307_sec5_v2_anchor_retune_metrics_{EVAL_SPLIT}",
                         index=False)

### §5b. v3 PR curve(whole split) 위에 v2의 실제 recall 표시 (v3 only)

In [ ]:
if VERSION == "v3":
    draw_pr(ev_all, MODELS, config.TARGET_PRECISION,
            f"{VERSION}_0307_sec5b_v2_anchor_pr_{EVAL_SPLIT}_all",
            marks=[(float(a["anchor_recall"]), f"v2 @{a['v2_tau']:g}: {float(a['anchor_recall']):.3f}",
                    "tab:red", "-.") for _, a in v2_anchors.iterrows()])
else:
    print("§5b only runs for VERSION='v3'.")